# VendorScope — build notebook (dev log)

These are my working notes from actually building this, kept close to how
I built it rather than cleaned up into a tutorial. I left the dead ends
in on purpose — if you're extending this, you'll probably hit the same
walls I did, and "here's the naive version, here's why it broke, here's
the fix" is more useful than a finished file with no history.

**How to use this:** run top to bottom. Sections 0–1 are research/notes,
no code. From Section 2 onward, `%%writefile` cells rebuild the actual
project on disk — same project as the clean submission package, just with
the reasoning left in.

Legend for the inline callouts below:
- 🔬 **Research note** — something I read up on / considered before writing code
- ❌ **Attempt that didn't work** — a real dead end, not a strawman
- ✅ **What actually shipped** — the version in the final code
- 🐛 **Bug found during testing** — caught by actually running it, not by inspection


## 0. Before writing any code

### Re-reading the brief

The one sentence that ended up shaping almost every design call:

> *"The LLM may judge proposal content, but it must not decide the final
> arithmetic, benchmark, tie-breaks, or rank."*

Everything downstream of that sentence follows pretty mechanically: exactly
one module gets to call an LLM (`tools/llm_tool.py`), and literally
everything else — validation, scoring, benchmarking, ranking, persistence
— has to be plain, boring, re-runnable Python. I kept coming back to this
line every time I was tempted to make something "smarter."

The rubric also quietly rewards this: Validation & scoring (20) +
Ranking & tie-breaks (20) + SQLite (10) = 50 of 100 points are about
deterministic, auditable code. "Agentic workflow" is only 20, and its own
description says "appropriate separation of LLM and tools" — not "most
agents wins."

### 🔬 Framework bake-off

The brief's own "suggested tools" column says *"Python functions or an
agent framework"* — so plain Python is the sanctioned baseline, and a
framework is optional flavor. I still wanted to try one, so here's what I
actually considered:

**AutoGen — built first, and it worked.** My first working second-engine
was AutoGen: an `Evaluator` `AssistantAgent` for the LLM call, and a
`ToolExecutor` `UserProxyAgent` that ran the deterministic validation/
ranking code through a real `code_execution_config` sandbox. This was a
legitimate, functioning implementation — I'm not describing a failure
here, I'm describing something I later *replaced*, which is a different
kind of "didn't survive contact with more thinking."

**Why I swapped it for LangGraph.** Once I actually looked at the brief's
own 10-step architecture diagram again — Setup → Input → Batch → Evaluate
→ Validate → Score → Benchmark → Rank → Persist → Present — I realized
that's *already a directed graph* with exactly one loop (evaluate-then-
validate, once per supplier). AutoGen models a *conversation between
agents*; LangGraph models *a graph*. The brief's diagram is a graph, not
a conversation. So LangGraph is a more literal implementation of the
document I was handed, not just "a different framework for variety."

**CrewAI — considered, rejected without building it.** CrewAI's whole
value proposition is multiple agents with roles/goals delegating to each
other autonomously. That's close to the opposite of "must not decide" —
I'd have had to invent personas for validation and ranking (tasks that
are supposed to have *zero* discretion) just to give CrewAI something to
delegate. Building it would have meant fighting the brief, not
implementing it. Didn't build it.

**Plain LangChain (no graph) — considered, rejected.** Its main offer here
would be structured-output parsers and PDF loaders. I already need
`tools/pdf_tool.py` and `tools/validation_tool.py` to be things a reviewer
can read start to finish without trusting a black-box `OutputParser`. Adding
LangChain would mean more dependency weight for less transparency, not more.

**RAG — considered, rejected.** RAG solves "the model can't see enough
relevant context in one prompt, so retrieve the relevant slice first."
Supplier proposals here are 2–4 pages; the whole thing fits in one prompt
with room to spare (the 12,000-char truncation in `build_prompt()` never
even triggers on the sample PDFs). There's no external corpus to retrieve
against either. Building RAG for this would be solving a problem the
brief doesn't have — the kind of scope creep that actually *costs* rubric
points ("appropriate" tool use is explicitly graded).

### 🔬 Robustness techniques: what to add, what to skip

Four things came up as "would make this more production-grade":

1. **Structured output enforcement** — worth doing. Providers can mostly
   guarantee schema-valid JSON at the API level now (`response_format`
   json_schema for OpenAI-compatible, forced tool-use for Anthropic).
   Free correctness, basically.
2. **Evidence-groundedness check** — worth doing, and cheap. After the
   LLM claims an `evidence` quote, check it actually appears in the real
   proposal text. Costs zero extra LLM calls.
3. **Retry/backoff on rate limits** — worth doing, standard hygiene.
4. **Self-consistency / majority voting** (call the LLM 2–3× per supplier,
   take the median) — **decided against.** It triples per-supplier token
   cost for a marginal robustness gain, which fights the token-budget
   goal directly. Documented as "considered, declined" rather than built.


## 1. Environment

One dependency note worth flagging: `langgraph` here is standalone — it
does NOT require the full `langchain` package. I almost installed
`langchain` out of habit before realizing `StateGraph`/`add_conditional_edges`/
`compile()` is all `langgraph` needs on its own.

In [ ]:
# streamlit        -- the UI
# pandas           -- dataframes for the leaderboard/scorecard tables
# pdfplumber       -- PDF text extraction (tools/pdf_tool.py)
# pydantic         -- schema validation for the LLM's scorecard JSON
# reportlab        -- generates the synthetic supplier PDFs
# anthropic/openai -- the two LLM client libraries (provider-resolved at runtime)
# langgraph        -- the graph-based orchestration engine (standalone, no langchain needed)
!pip install -q streamlit pandas pdfplumber "pydantic>=2.5.0" reportlab anthropic openai langgraph


### Project scaffold

In [ ]:
import os
for d in ["vendorscope/database", "vendorscope/tools", "vendorscope/agents",
          "vendorscope/agents_langgraph", "vendorscope/sample_data/supplier_pdfs",
          "vendorscope/sample_output"]:
    os.makedirs(d, exist_ok=True)
print("Folders created.")


## 2. Database layer

Nothing dramatic here — SQLite is what the brief asks for, and a 5-row
criteria table plus two log tables doesn't need anything heavier. The one
decision worth noting: `init_db()` only seeds default criteria if the
table is empty, so calling it repeatedly (e.g. every Streamlit rerun)
never clobbers criteria you've edited by hand in the DB.

In [ ]:
%%writefile vendorscope/database/__init__.py


In [ ]:
%%writefile vendorscope/database/db_setup.py
"""
db_setup.py
Creates the SQLite database and seeds it with default evaluation criteria.
Run standalone:  python database/db_setup.py
Or imported by app.py to guarantee the DB exists before Streamlit starts.
"""

import sqlite3
import os

DB_PATH = os.path.join(os.path.dirname(__file__), "rfp_evaluation.db")

SCHEMA = """
CREATE TABLE IF NOT EXISTS evaluation_criteria (
    criterion_id    INTEGER PRIMARY KEY AUTOINCREMENT,
    name            TEXT NOT NULL,
    description     TEXT,
    weight          REAL NOT NULL,      -- percentage, e.g. 30 means 30%
    max_score       INTEGER NOT NULL DEFAULT 10,
    is_active       INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE IF NOT EXISTS rfp_runs (
    rfp_run_id      TEXT PRIMARY KEY,   -- UUID
    created_at      TEXT NOT NULL,
    status          TEXT NOT NULL DEFAULT 'created'
);

CREATE TABLE IF NOT EXISTS supplier_results (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    rfp_run_id          TEXT NOT NULL,
    supplier_name       TEXT NOT NULL,
    submission_date     TEXT NOT NULL,
    experience_rating   REAL NOT NULL,
    absolute_score      REAL,
    ppi                 REAL,
    final_rank          INTEGER,
    result_json         TEXT,
    FOREIGN KEY (rfp_run_id) REFERENCES rfp_runs (rfp_run_id)
);
"""

DEFAULT_CRITERIA = [
    ("Technical Capability", "Architecture, integrations, scalability, technical fit", 30, 10, 1),
    ("Implementation Plan", "Timeline, milestones, staffing, risk plan", 20, 10, 1),
    ("Commercial Value", "Pricing clarity, total cost, assumptions", 20, 10, 1),
    ("Security & Compliance", "Controls, certifications, privacy, auditability", 20, 10, 1),
    ("Support & Experience", "Support model, similar projects, references", 10, 10, 1),
]


def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


def init_db(reseed_if_empty=True):
    """Creates tables if they don't exist and seeds default criteria if the
    criteria table is empty. Safe to call every app startup."""
    conn = get_connection()
    cur = conn.cursor()
    cur.executescript(SCHEMA)
    conn.commit()

    if reseed_if_empty:
        cur.execute("SELECT COUNT(*) AS c FROM evaluation_criteria")
        count = cur.fetchone()["c"]
        if count == 0:
            cur.executemany(
                """INSERT INTO evaluation_criteria
                   (name, description, weight, max_score, is_active)
                   VALUES (?, ?, ?, ?, ?)""",
                DEFAULT_CRITERIA,
            )
            conn.commit()
    conn.close()


def get_active_criteria():
    conn = get_connection()
    rows = conn.execute(
        "SELECT * FROM evaluation_criteria WHERE is_active = 1 ORDER BY criterion_id"
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def active_weight_total():
    return sum(c["weight"] for c in get_active_criteria())


if __name__ == "__main__":
    init_db()
    print(f"Database ready at {DB_PATH}")
    crit = get_active_criteria()
    print(f"{len(crit)} active criteria, total weight = {active_weight_total()}%")
    for c in crit:
        print(f"  - {c['name']} ({c['weight']}%)")


## 3. The tools — where most of the actual iteration happened

### 3a. Document Tool — no drama

Extracts text from the PDF, in memory, no temp files. This one worked on
the first pass and stayed that way.

In [ ]:
%%writefile vendorscope/tools/__init__.py


In [ ]:
%%writefile vendorscope/tools/pdf_tool.py
"""
pdf_tool.py
Document Tool: extracts clean text from an uploaded supplier RFP PDF.
"""

import pdfplumber
import io


def extract_text_from_pdf(file_like) -> str:
    """
    Accepts a file path (str) or a file-like object (e.g. Streamlit's
    UploadedFile, or a BytesIO buffer) and returns concatenated, cleaned
    plain text from every page.
    """
    text_parts = []

    if isinstance(file_like, (bytes, bytearray)):
        file_like = io.BytesIO(file_like)
    elif hasattr(file_like, "read") and not hasattr(file_like, "seek"):
        file_like = io.BytesIO(file_like.read())

    if hasattr(file_like, "seek"):
        file_like.seek(0)

    with pdfplumber.open(file_like) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            text_parts.append(page_text)

    full_text = "\n".join(text_parts)
    # Light cleanup: collapse excessive whitespace but keep line structure
    lines = [ln.strip() for ln in full_text.splitlines()]
    lines = [ln for ln in lines if ln]  # drop empty lines
    return "\n".join(lines)


def extract_text_from_path(path: str) -> str:
    with open(path, "rb") as f:
        return extract_text_from_pdf(f)


### 3b. Evaluation Agent — this one went through real revisions

**❌ Attempt 1 — just ask nicely.** The first version of the prompt ended
with "output ONLY valid JSON" and called it done:

```python
# ATTEMPT 1 (not what shipped) -- illustrative only, not run in this notebook
response = client.chat.completions.create(
    model=model, messages=[{"role": "user", "content": prompt}],
)
raw_json = response.choices[0].message.content
```

This mostly worked. "Mostly" is the problem — often enough, the model
would wrap the JSON in a markdown fence, add a stray sentence before it,
or occasionally just drop a field. All of that got caught by
`validation_tool.py` downstream, but it meant more warnings firing for
reasons that had nothing to do with the *evaluation quality* — just
formatting noise.

**✅ What shipped — structured output enforcement.** OpenAI-compatible
providers (OpenRouter/OpenAI) support `response_format={"type":
"json_schema", ...}`, which constrains the model's *sampling*, not just
the prompt wording — genuinely different from asking nicely. Anthropic
doesn't have that exact parameter, but forcing a tool-use call
(`tool_choice={"type": "tool", "name": "submit_scorecard"}`) gets the same
effect: the model's response comes back as already-parsed, schema-shaped
arguments.

One thing I had to handle: **not every model/provider honors
`response_format`.** Some OpenRouter backend models reject it outright.
So the shipped version tries the schema-constrained call first and
transparently retries once, plainly, if that raises — see
`_call_openai_compatible()` below. `tools/validation_tool.py` still
re-validates everything regardless of which path was taken — a schema
constrains *shape*, not whether the values are sensible, so this is
defense in depth, not a replacement for validation.

**🔬 Also added: retry/backoff.** Hit a 429 mid-testing-batch and the
whole run died on one transient rate limit. `_with_retries()` gives
retryable errors (429/5xx) up to 2 extra attempts with exponential
backoff, and lets non-retryable errors (bad key, malformed request) fail
immediately rather than wasting time retrying something that will never
succeed.

In [ ]:
# --- Evaluation Agent, ATTEMPT 1 (didn't ship) ---
#
# Simplest possible version: ask the model nicely, trust the prompt wording.
#
# from openai import OpenAI
# client = OpenAI(api_key=api_key, base_url=base_url)
# response = client.chat.completions.create(
#     model=model,
#     max_tokens=800,
#     messages=[{"role": "user", "content": prompt}],  # prompt just SAYS "output JSON only"
# )
# raw_json = response.choices[0].message.content
#
# Testing this against the real synthetic proposals surfaced two separate
# problems:
#
#   1. Formatting noise: some responses wrapped the JSON in ```json ...```
#      fences, occasionally a criterion was silently dropped. Not
#      catastrophic -- tools/validation_tool.py already catches malformed
#      JSON and missing criteria -- but it meant warnings were firing for
#      reasons that had nothing to do with evaluation quality, just
#      response formatting the prompt wording didn't fully control.
#
#   2. No resilience: one 429 (rate limit) mid-batch killed the entire
#      run, no retry, no backoff.
#
# --- WHAT SHIPPED: see tools/llm_tool.py below ---
# (1) is fixed by structured output enforcement -- response_format=
# json_schema for OpenAI-compatible providers, forced tool-use for
# Anthropic (_call_openai_compatible / _call_anthropic below). (2) is
# fixed by _with_retries() -- up to 2 retries with exponential backoff on
# 429/5xx, immediate failure on non-retryable errors. Validation still
# re-checks everything regardless of which path produced it.
print("(no-op cell -- see tools/llm_tool.py below for what shipped)")


In [ ]:
%%writefile vendorscope/tools/llm_tool.py
"""
llm_tool.py
Evaluation Agent: sends one supplier's proposal text + the active criteria
list to an LLM and asks for a strict-JSON, evidence-grounded scorecard.

The LLM is ONLY allowed to judge proposal content (scores, justification,
evidence, risks). It must NEVER compute weighted totals, benchmarks,
tie-breaks, or ranks -- that is done later by ranking_tool.py in pure
deterministic Python.

This module always makes a REAL LLM call. Provider resolution mirrors the
earlier AutoGen coding-agent project's convention:
  1. explicit api_key/base_url arguments
  2. OPENROUTER_API_KEY env var -> OpenRouter (recommended; one key, many
     models, free-tier options for a class project)
  3. ANTHROPIC_API_KEY env var -> direct Anthropic call
  4. OPENAI_API_KEY env var -> direct OpenAI call
If none of those resolve, resolve_provider() raises a clear ValueError --
there is no offline simulator to fall back to.

Robustness techniques (all stay within "LLM judges content, Python does
everything else" -- none of this changes what the LLM is allowed to
decide):
  - Structured output enforcement: OpenAI-compatible calls request
    response_format=json_schema (guarantees schema-valid JSON from
    providers that support it); Anthropic calls force a tool-use call
    against the same schema. Either way, tools/validation_tool.py still
    re-validates everything -- never trust the wire even with a schema.
  - Retry/backoff: transient rate-limit/server errors get up to 2 retries
    with exponential backoff before giving up.

Deliberately NOT implemented: self-consistency / majority voting (calling
the LLM 2-3x per supplier and taking the median score). It would triple
the per-supplier token/credit cost for a marginal robustness gain, which
cuts against this project's own token-budget goals -- noted here as a
technique considered and declined, not one worth building for this scope.
"""

import os
import json
import time

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DEFAULT_MODEL_OPENROUTER = "openai/gpt-4o-mini"
DEFAULT_MODEL_OPENAI = "gpt-4o-mini"
DEFAULT_MODEL_ANTHROPIC = "claude-sonnet-4-6"
# max_tokens is capped to avoid OpenRouter 402 "insufficient credits"
# errors on free/low-balance accounts, which are triggered by the
# request's token budget, not actual usage.
DEFAULT_MAX_TOKENS = 800

MAX_RETRIES = 2          # additional attempts after the first, so 3 total
RETRY_BASE_DELAY = 1.5   # seconds; doubles each retry (1.5s, 3s)

# JSON Schema the LLM's scorecard must conform to. Used for structured
# output on OpenAI-compatible providers (response_format) and as the
# input_schema for a forced tool call on Anthropic. This is belt-and-
# braces with tools/validation_tool.py, not a replacement for it -- a
# schema constrains shape, not whether values are sensible, and providers
# that don't honor response_format still fall back to a plain call.
SCORECARD_SCHEMA = {
    "type": "object",
    "properties": {
        "supplier_name": {"type": "string"},
        "criteria": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "criterion_id": {"type": "integer"},
                    "score": {"type": "number"},
                    "max_score": {"type": "number"},
                    "justification": {"type": "string"},
                    "evidence": {"type": "string"},
                },
                "required": ["criterion_id", "score", "max_score", "justification", "evidence"],
            },
        },
        "risks": {"type": "array", "items": {"type": "string"}},
        "overall_summary": {"type": "string"},
    },
    "required": ["supplier_name", "criteria", "risks", "overall_summary"],
}


def build_prompt(supplier_name: str, proposal_text: str, criteria: list) -> str:
    criteria_lines = "\n".join(
        f'- criterion_id={c["criterion_id"]}, name="{c["name"]}", '
        f'max_score={c["max_score"]}, focus="{c["description"]}"'
        for c in criteria
    )
    return f"""You are an impartial procurement evaluator.

Supplier: {supplier_name}

Active evaluation criteria (you must return exactly one result for every
criterion listed, using the same criterion_id):
{criteria_lines}

Supplier proposal text (use ONLY evidence found in this text; do not invent
facts that are not present):
\"\"\"
{proposal_text[:12000]}
\"\"\"

Instructions:
1. Score each criterion from 0 to its max_score based only on evidence in
   the text above.
2. Provide a short justification and a direct quote/paraphrase as evidence
   for each score. The evidence must be text that actually appears in the
   proposal above -- do not paraphrase so loosely that it no longer
   matches the source wording.
3. List any notable risks you observe.
4. Provide a one or two sentence overall_summary.
5. Output ONLY valid JSON, no markdown fences, no commentary, matching
   exactly this schema:

{{
  "supplier_name": "string",
  "criteria": [
    {{"criterion_id": int, "score": number, "max_score": number,
      "justification": "string", "evidence": "string"}}
  ],
  "risks": ["string", ...],
  "overall_summary": "string"
}}
"""


def resolve_provider(model: str = None, api_key: str = None, base_url: str = None,
                      provider: str = None) -> dict:
    """
    Resolves which LLM backend a call should use, in priority order:
      1. explicit api_key (+ optional base_url / model / provider) passed by the caller
      2. OPENROUTER_API_KEY env var  -> OpenRouter, model defaults to
         "openai/gpt-4o-mini" (OpenRouter's provider-prefixed naming)
      3. ANTHROPIC_API_KEY env var   -> direct Anthropic call
      4. OPENAI_API_KEY env var      -> direct OpenAI call

    provider: only meaningful together with an explicit api_key -- hints
    which backend that key belongs to: "openrouter" (default if omitted),
    "anthropic", or "openai". Without this hint, an explicit api_key is
    assumed to be an OpenRouter key (this project's recommended provider),
    which would be wrong for an explicitly-Anthropic or -OpenAI key -- so
    any caller accepting a user-chosen provider alongside a user-entered
    key (e.g. a UI with a provider dropdown) should always pass this.

    Returns {"provider": "anthropic" | "openai_compatible", "api_key": ...,
             "model": ..., "base_url": ... (only for openai_compatible)}

    Raises ValueError if none resolve -- there is no fallback.
    """
    if api_key:
        if provider == "anthropic":
            return {
                "provider": "anthropic",
                "api_key": api_key,
                "model": model or DEFAULT_MODEL_ANTHROPIC,
            }
        if provider == "openai":
            return {
                "provider": "openai_compatible",
                "api_key": api_key,
                "base_url": base_url,  # None -> official OpenAI endpoint
                "model": model or DEFAULT_MODEL_OPENAI,
            }
        # provider == "openrouter" or unspecified: default assumption
        return {
            "provider": "openai_compatible",
            "api_key": api_key,
            "base_url": base_url or OPENROUTER_BASE_URL,
            "model": model or DEFAULT_MODEL_OPENROUTER,
        }

    openrouter_key = os.environ.get("OPENROUTER_API_KEY")
    if openrouter_key:
        return {
            "provider": "openai_compatible",
            "api_key": openrouter_key,
            "base_url": base_url or os.environ.get("OPENAI_API_BASE") or OPENROUTER_BASE_URL,
            "model": model or DEFAULT_MODEL_OPENROUTER,
        }

    anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
    if anthropic_key:
        return {
            "provider": "anthropic",
            "api_key": anthropic_key,
            "model": model or DEFAULT_MODEL_ANTHROPIC,
        }

    openai_key = os.environ.get("OPENAI_API_KEY")
    if openai_key:
        return {
            "provider": "openai_compatible",
            "api_key": openai_key,
            "base_url": base_url or os.environ.get("OPENAI_API_BASE"),  # None -> official OpenAI endpoint
            "model": model or DEFAULT_MODEL_OPENAI,
        }

    raise ValueError(
        "No LLM API key found. Set one of: OPENROUTER_API_KEY "
        "(recommended -- one key, many models), ANTHROPIC_API_KEY, or "
        "OPENAI_API_KEY as an environment variable, or pass api_key=... "
        "explicitly."
    )


def _is_retryable(exc: Exception) -> bool:
    """True for rate-limit / transient-server errors worth retrying.
    Checked defensively (by status code / class name) rather than
    importing every provider SDK's specific exception classes, since both
    openai and anthropic expose slightly different hierarchies across
    versions.

    IMPORTANT: only matches on the SPECIFIC subclass names for rate-limit
    and server errors -- never on the generic base class ("APIStatusError"
    for openai, "APIError" for anthropic). That base class is what gets
    raised for errors that have no more specific subclass -- including
    401 (bad key), 402 (insufficient credits/payment required), 403
    (forbidden), and 404 (model not found). None of those are fixed by
    retrying; matching the base class name here would silently waste time
    retrying a permanent failure before it finally surfaces to the caller.
    """
    status = getattr(exc, "status_code", None)
    if status in (429, 500, 502, 503, 529):
        return True
    name = type(exc).__name__
    return "RateLimitError" in name or "InternalServerError" in name or "APIConnectionError" in name


def _with_retries(fn, *args, **kwargs):
    """Runs fn(*args, **kwargs) with up to MAX_RETRIES extra attempts on
    retryable errors, using exponential backoff. Non-retryable errors
    (bad API key, malformed request, etc.) raise immediately."""
    attempt = 0
    while True:
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            if attempt >= MAX_RETRIES or not _is_retryable(exc):
                raise
            time.sleep(RETRY_BASE_DELAY * (2 ** attempt))
            attempt += 1


def _call_anthropic(cfg: dict, prompt: str, tokens: int) -> str:
    """Forces a structured tool-use call so Anthropic returns the
    scorecard as already-parsed, schema-shaped arguments rather than
    free-form text. tools/validation_tool.py still re-validates the
    result -- this only reduces malformed-JSON noise at the source."""
    import anthropic
    client = anthropic.Anthropic(api_key=cfg["api_key"])

    def _do_call():
        return client.messages.create(
            model=cfg["model"],
            max_tokens=tokens,
            tools=[{
                "name": "submit_scorecard",
                "description": "Submit the completed supplier evaluation scorecard.",
                "input_schema": SCORECARD_SCHEMA,
            }],
            tool_choice={"type": "tool", "name": "submit_scorecard"},
            messages=[{"role": "user", "content": prompt}],
        )

    response = _with_retries(_do_call)
    for block in response.content:
        if block.type == "tool_use" and block.name == "submit_scorecard":
            return json.dumps(block.input)
    # Fallback: no tool_use block found (shouldn't happen with a forced
    # tool_choice, but don't crash if a future API version changes shape) --
    # return whatever text came back so validation_tool can attempt to
    # parse/flag it through the normal error-handling path.
    return "".join(b.text for b in response.content if getattr(b, "type", None) == "text")


def _call_openai_compatible(cfg: dict, prompt: str, tokens: int) -> str:
    """Requests structured JSON output via response_format when the
    provider/model supports it, and transparently falls back to a plain
    call if the provider rejects that parameter (not every OpenRouter
    backend model honors response_format)."""
    from openai import OpenAI
    client = OpenAI(api_key=cfg["api_key"], base_url=cfg.get("base_url"))
    extra_headers = None
    if cfg.get("base_url") == OPENROUTER_BASE_URL:
        extra_headers = {"X-Title": "VendorScope"}

    def _do_call(use_schema: bool):
        kwargs = dict(
            model=cfg["model"],
            max_tokens=tokens,
            temperature=0,
            messages=[{"role": "user", "content": prompt}],
            extra_headers=extra_headers,
        )
        if use_schema:
            kwargs["response_format"] = {
                "type": "json_schema",
                "json_schema": {"name": "scorecard", "schema": SCORECARD_SCHEMA, "strict": False},
            }
        return client.chat.completions.create(**kwargs)

    try:
        response = _with_retries(_do_call, use_schema=True)
    except Exception:
        # Provider/model doesn't support response_format (or some other
        # request-shape issue) -- retry once, plainly, before giving up.
        response = _with_retries(_do_call, use_schema=False)
    return response.choices[0].message.content


def call_llm_live(prompt: str, model: str = None, api_key: str = None,
                   base_url: str = None, max_tokens: int = None,
                   provider: str = None) -> str:
    """Makes one real LLM call and returns the raw text response.
    max_tokens caps the OUTPUT length of this single call (defaults to
    DEFAULT_MAX_TOKENS=800, comfortably enough for a 5-criterion JSON
    scorecard with short justifications) -- lower it further if you're on
    a tight token/credit budget, e.g. max_tokens=500. provider is only
    meaningful together with an explicit api_key -- see resolve_provider().
    """
    cfg = resolve_provider(model=model, api_key=api_key, base_url=base_url, provider=provider)
    tokens = max_tokens if max_tokens is not None else DEFAULT_MAX_TOKENS

    if cfg["provider"] == "anthropic":
        return _call_anthropic(cfg, prompt, tokens)
    return _call_openai_compatible(cfg, prompt, tokens)


def evaluate_supplier(supplier_name: str, proposal_text: str, criteria: list,
                       model: str = None, api_key: str = None,
                       base_url: str = None, max_tokens: int = None,
                       provider: str = None) -> str:
    """
    Builds the evaluation prompt and makes a real LLM call, returning the
    raw JSON string response (not yet parsed/validated -- that's
    tools/validation_tool.py's job). model/api_key/base_url/max_tokens/
    provider are passed straight to resolve_provider()/call_llm_live();
    leave them None to resolve purely from environment variables. Raises
    ValueError if no API key resolves.
    """
    prompt = build_prompt(supplier_name, proposal_text, criteria)
    return call_llm_live(prompt, model=model, api_key=api_key,
                          base_url=base_url, max_tokens=max_tokens, provider=provider)


In [ ]:
# --- Bug found AFTER shipping, in an actual Streamlit Cloud deployment ---
#
# Real crash, not a hypothetical:
#
#   openai.APIStatusError: This app has encountered an error...
#   File ".../tools/llm_tool.py", line 300, in _call_openai_compatible
#       response = _with_retries(_do_call, use_schema=False)
#   File ".../openai/_base_client.py", line 1141, in request
#       raise self._make_status_error_from_response(err.response) from None
#
# ATTEMPT 1 for _is_retryable() (shipped, then found broken):
#
#   def _is_retryable(exc):
#       status = getattr(exc, "status_code", None)
#       if status in (429, 500, 502, 503, 529):
#           return True
#       name = type(exc).__name__
#       return "RateLimit" in name or "APIStatusError" in name or "InternalServerError" in name
#                                      ^^^^^^^^^^^^^^^^^^^^^^^^
#                                      the bug is right here
#
# `APIStatusError` is openai's GENERIC BASE CLASS -- it's what gets raised
# for any error that doesn't have a more specific subclass, which includes
# 401 (bad key), 402 (payment required / insufficient credits), 403, and
# 404 (model not found). None of those are fixed by retrying. Matching on
# the base class name meant a permanent failure (in this real case: an
# OpenRouter account with a $0 balance calling a PAID model,
# "openai/gpt-4o-mini", which isn't one of OpenRouter's free-tier models)
# got retried up to 3 times, then the fallback path retried it 3 MORE
# times, wasting several seconds of backoff sleeps before finally
# surfacing the exact same unfixable error to the user.
#
# FIX (what shipped -- see tools/llm_tool.py above): drop the base-class
# match entirely; only match the SPECIFIC subclass names
# (RateLimitError, InternalServerError, APIConnectionError) plus the
# explicit status-code list. Verified with fake exceptions carrying each
# status code (401/402/429/500) to confirm client errors now fail on
# attempt 1, and genuine rate limits/server errors still retry correctly.
#
# Practical fix for the actual deployment: the default OpenRouter model
# (openai/gpt-4o-mini) is paid, not free -- either add OpenRouter credits,
# or use the new "Model override" field in the app's Advanced settings to
# point at a genuine free model instead (e.g. a current ":free"-suffixed
# model from openrouter.ai/models).
print("(no-op cell -- see the fixed _is_retryable() in tools/llm_tool.py above)")


### 3c. Validation Tool — the groundedness check was an afterthought that turned out to matter

**❌ First cut** only handled the two error modes the brief explicitly
names: out-of-range scores (clip to max) and missing criteria (default to
0, flag it). Both are still in there, unchanged.

**🔬 Then I actually read a few LLM responses closely.** The scores looked
fine, the *evidence quotes* didn't always. A model will happily produce a
very confident, specific-sounding "quote" that isn't actually in the
document — not maliciously, just because generating a plausible-sounding
sentence is what these models are good at, and "plausible" isn't the same
as "true." Score clipping doesn't catch this at all; a fabricated quote
can have a perfectly in-range score attached to it.

**✅ What shipped — `is_evidence_grounded()`.** Cheap, not RAG (there's
nothing to *retrieve* — the whole proposal's already in the prompt), just
a post-hoc sanity check: is the claimed evidence a literal substring of
the proposal text, or does it at least share most of its significant
words with something in the document? If neither, flag it. Doesn't touch
the score — there's no sensible way to "clip" a fabricated quote, only to
flag it for a human to look at.

In [ ]:
# --- Evidence-groundedness check, ATTEMPT 1 (didn't ship) ---
#
# First idea: exact substring match, nothing else.
#
# def is_evidence_grounded_v1(evidence, proposal_text):
#     return evidence.strip().lower() in proposal_text.lower()
#
# Tried this against real LLM output and it failed constantly -- not
# because the model was lying, but because it paraphrases even when
# telling the truth. Proposal text: "SOC 2 Type II and ISO 27001
# certifications." Model's evidence field: "the supplier holds SOC 2 Type
# II and ISO 27001 certs." That's an honest answer. Exact substring match
# flags it as ungrounded anyway, because "certs" != "certifications" and
# the word order/wrapping differs. Running this version would have
# produced a warning on almost every criterion, for almost every supplier
# -- false positives so common they'd drown out the rare real ones,
# which is worse than not checking at all.
#
# --- WHAT SHIPPED: is_evidence_grounded() below ---
# Substring match first (still catches direct quotes, cheaply), and only
# if that fails, a word-overlap fallback: strip stopwords, check what
# fraction of the evidence's significant words appear anywhere in the
# proposal text. Honest paraphrases pass; evidence sharing no real
# vocabulary with the source document gets flagged.
print("(no-op cell -- see is_evidence_grounded() in the file below for what shipped)")


In [ ]:
%%writefile vendorscope/tools/validation_tool.py
"""
validation_tool.py
Validation Tool: parses the raw LLM JSON string, checks it against a strict
schema, fills in missing criteria, clips out-of-range scores, checks that
claimed evidence is actually grounded in the source proposal text, and
records warnings for all of the above. This is the ONLY place malformed
or ungrounded LLM output gets fixed/flagged -- once data passes through
here, ranking_tool.py can trust it completely.
"""

import json
import re
from typing import List, Optional
from pydantic import BaseModel, ValidationError, field_validator

# Common short words excluded from the evidence-groundedness word-overlap
# check below, so grounding isn't judged on articles/prepositions that
# would trivially "match" almost any proposal text.
_STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for", "with",
    "is", "are", "this", "that", "by", "as", "at", "from", "its", "it",
    "was", "were", "be", "been", "has", "have", "had", "will", "their",
}


class CriterionResult(BaseModel):
    criterion_id: int
    score: float
    max_score: float
    justification: str = ""
    evidence: str = ""

    @field_validator("score")
    @classmethod
    def score_non_negative(cls, v):
        return max(0.0, v)


class LLMScorecard(BaseModel):
    supplier_name: str
    criteria: List[CriterionResult]
    risks: Optional[List[str]] = []
    overall_summary: Optional[str] = ""


def _normalize_text(t: str) -> str:
    return re.sub(r"\s+", " ", t.lower()).strip()


def is_evidence_grounded(evidence: str, proposal_text: str, min_word_overlap: float = 0.5) -> bool:
    """
    Cheap, RAG-adjacent (but not RAG) groundedness check: does the LLM's
    claimed 'evidence' string actually appear in the source proposal text,
    or at least substantially overlap with it? This is deliberately not a
    retrieval system -- there's nothing to retrieve, the whole proposal is
    already in the prompt -- it's a post-hoc sanity check that the model
    didn't fabricate a quote.

    Returns True (grounded) if either:
      - the normalized evidence string is a literal substring of the
        normalized proposal text, or
      - at least `min_word_overlap` fraction of the evidence's
        significant words (length > 3, not a stopword) appear anywhere
        in the proposal text.

    Returns True (i.e. does not flag) when there's nothing meaningful to
    check -- empty evidence/proposal, or an evidence string with no
    significant words -- since there's no basis to call it "ungrounded"
    in that case.
    """
    if not evidence or not proposal_text:
        return True

    norm_evidence = _normalize_text(evidence)
    norm_proposal = _normalize_text(proposal_text)

    if norm_evidence in norm_proposal:
        return True

    words = [w for w in re.findall(r"[a-z0-9]+", norm_evidence)
             if len(w) > 3 and w not in _STOPWORDS]
    if not words:
        return True

    present = sum(1 for w in words if w in norm_proposal)
    return (present / len(words)) >= min_word_overlap


def validate_and_normalize(raw_json: str, active_criteria: list, supplier_name: str,
                            proposal_text: str = None):
    """
    Returns (normalized_criteria: list[dict], warnings: list[str])

    normalized_criteria has exactly one entry per active criterion, with:
      criterion_id, name, weight, max_score, score (clipped 0..max_score),
      justification, evidence

    proposal_text: optional. When given, each criterion's claimed evidence
    is checked against it with is_evidence_grounded() and a warning is
    recorded (not a score change -- there's no sensible "clip" for
    fabricated text) if it doesn't appear to be grounded. Omit this
    argument to skip the groundedness check entirely (e.g. if the caller
    doesn't have the proposal text on hand for some reason).
    """
    warnings = []
    parsed_map = {}  # criterion_id -> CriterionResult

    # --- Step 1: parse JSON safely ---
    try:
        # Strip accidental markdown fences some LLMs add
        cleaned = raw_json.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.strip("`")
            if cleaned.lower().startswith("json"):
                cleaned = cleaned[4:]
        data = json.loads(cleaned)
    except (json.JSONDecodeError, TypeError) as e:
        warnings.append(f"Could not parse LLM JSON output ({e}). "
                         f"All criteria defaulted to 0.")
        data = {"supplier_name": supplier_name, "criteria": [], "risks": [], "overall_summary": ""}

    # --- Step 2: schema validation ---
    try:
        scorecard = LLMScorecard(**data)
        for cr in scorecard.criteria:
            parsed_map[cr.criterion_id] = cr
    except ValidationError as e:
        warnings.append(f"Schema validation issues: {e.error_count()} field error(s) "
                         f"detected; affected entries were skipped.")
        # Best-effort: pull whatever criteria entries do individually validate
        for item in data.get("criteria", []):
            try:
                cr = CriterionResult(**item)
                parsed_map[cr.criterion_id] = cr
            except (ValidationError, TypeError):
                continue

    # --- Step 3: reconcile against the active criteria list (source of truth) ---
    normalized = []
    for c in active_criteria:
        cid = c["criterion_id"]
        max_score = c["max_score"]

        if cid in parsed_map:
            cr = parsed_map[cid]
            score = cr.score
            justification = cr.justification
            evidence = cr.evidence

            # clip out-of-range scores
            if score > max_score:
                warnings.append(
                    f'"{c["name"]}": LLM score {score} exceeded max_score '
                    f"{max_score}; clipped to {max_score}."
                )
                score = max_score
            if cr.max_score != max_score:
                warnings.append(
                    f'"{c["name"]}": LLM reported max_score {cr.max_score}, '
                    f"expected {max_score}; using configured max_score."
                )

            # evidence-groundedness check (only if proposal_text was given)
            if proposal_text is not None and evidence and not is_evidence_grounded(evidence, proposal_text):
                warnings.append(
                    f'"{c["name"]}": claimed evidence does not appear to be '
                    f"grounded in the supplier's proposal text; flagged for "
                    f"manual review."
                )
        else:
            warnings.append(
                f'"{c["name"]}": missing from LLM output; defaulted to 0 '
                f"and flagged for manual review."
            )
            score = 0.0
            justification = "MISSING - not returned by LLM"
            evidence = ""

        normalized.append({
            "criterion_id": cid,
            "name": c["name"],
            "weight": c["weight"],
            "max_score": max_score,
            "score": round(score, 2),
            "justification": justification,
            "evidence": evidence,
        })

    return normalized, warnings


### 3d. Ranking Tool — deliberately the most boring file in the project

This is where I *wanted* zero surprises, and got them. Every formula here
is a direct implementation of the brief's own spec (absolute weighted
score, benchmark, gap, relative %, PPI, the 4-level tie-break). I verified
this one differently than the others: not by finding bugs, but by
proving there *weren't* any — ran the ranking pipeline 30 times (10 fixed
supplier order, 20 randomly shuffled) on identical input and diffed the
JSON output. Byte-identical every time. That's the actual point of this
file existing separately from everything that touches an LLM.

In [ ]:
%%writefile vendorscope/tools/ranking_tool.py
"""
ranking_tool.py
Ranking Tool: pure deterministic Python. No LLM calls happen here.
Implements exactly the formulas and tie-break order from the project brief.
"""

from datetime import datetime


def compute_absolute_score(normalized_criteria: list) -> float:
    """Sum of (criterion score / max_score) * weight, across all criteria."""
    total = 0.0
    for c in normalized_criteria:
        if c["max_score"] > 0:
            total += (c["score"] / c["max_score"]) * c["weight"]
    return round(total, 4)


def compute_benchmarks(all_suppliers_criteria: dict) -> dict:
    """
    all_suppliers_criteria: {supplier_name: [normalized_criteria...]}
    Returns {criterion_id: benchmark_score} = highest valid score observed
    for that criterion across all suppliers.
    """
    benchmarks = {}
    for supplier, criteria in all_suppliers_criteria.items():
        for c in criteria:
            cid = c["criterion_id"]
            benchmarks[cid] = max(benchmarks.get(cid, 0.0), c["score"])
    return benchmarks


def compute_gaps_and_relative(normalized_criteria: list, benchmarks: dict) -> list:
    """Adds 'benchmark', 'gap', and 'relative_pct' to each criterion dict."""
    enriched = []
    for c in normalized_criteria:
        cid = c["criterion_id"]
        benchmark = benchmarks.get(cid, 0.0)
        gap = round(c["score"] - benchmark, 2)

        if benchmark == 0:
            # safe handling: no valid peer signal for this criterion
            relative_pct = 100.0 if c["score"] == 0 else 0.0
        else:
            relative_pct = round((c["score"] / benchmark) * 100, 2)

        enriched.append({**c, "benchmark": benchmark, "gap": gap, "relative_pct": relative_pct})
    return enriched


def compute_ppi(enriched_criteria: list) -> float:
    """Weighted average of criterion relative-performance percentages."""
    total_weight = sum(c["weight"] for c in enriched_criteria)
    if total_weight == 0:
        return 0.0
    weighted_sum = sum(c["relative_pct"] * c["weight"] for c in enriched_criteria)
    return round(weighted_sum / total_weight, 2)


def _parse_date(d):
    if isinstance(d, str):
        return datetime.fromisoformat(d)
    return d


def rank_suppliers(supplier_records: list) -> list:
    """
    supplier_records: list of dicts, each with:
        supplier_name, submission_date (ISO str), experience_rating,
        ppi, absolute_score, criteria (enriched list)

    Applies the mandatory tie-break order:
      1) Higher PPI first
      2) Earlier submission date
      3) Higher historical experience rating
      4) Supplier name ascending
    Then assigns sequential final_rank 1, 2, 3...
    """
    def sort_key(r):
        return (
            -r["ppi"],                          # higher PPI first
            _parse_date(r["submission_date"]),   # earlier date first
            -r["experience_rating"],             # higher experience first
            r["supplier_name"].lower(),          # name ascending
        )

    ranked = sorted(supplier_records, key=sort_key)
    for i, r in enumerate(ranked, start=1):
        r["final_rank"] = i
    return ranked


## 4. Orchestration — two engines, same tools

### 4a. Direct engine

Plain function calls, in the order the brief's diagram specifies. This
was always the reference implementation — if the LangGraph engine ever
disagreed with this one on the same input, I'd trust this one and go
hunting for the bug in the graph, not the other way around.

In [ ]:
%%writefile vendorscope/agents/__init__.py


In [ ]:
%%writefile vendorscope/agents/orchestrator.py
"""
orchestrator.py
Orchestrator Agent (Direct engine): controls the workflow end-to-end,
calling each tool in the required order with plain Python function calls
-- no agent framework. This module has no UI code -- Streamlit (app.py)
only calls run_batch_evaluation() and renders whatever it returns.

Makes a real LLM call for every supplier -- there is no mock/offline mode
anywhere in this pipeline. If no API key resolves (see
tools.llm_tool.resolve_provider()), run_batch_evaluation() raises a
ValueError immediately, before creating a run row or touching the database.

Robustness techniques applied here (see tools/llm_tool.py and
tools/validation_tool.py for the implementations): structured-output
enforcement on the LLM call, evidence-groundedness checking on the
Validation Tool, and retry/backoff on transient LLM errors. All three stay
within "LLM judges content, Python does everything else" -- none change
what the LLM is allowed to decide.

Pipeline (matches the brief's 10-step architecture):
  1. Setup      -> criteria already loaded by caller
  2. Input      -> caller supplies supplier_inputs (name, date, experience, pdf bytes)
  3. Batch      -> create rfp_run row / run id
  4. Evaluate   -> extract text, build prompt, call LLM   (per supplier)
  5. Validate   -> parse + normalize LLM JSON              (per supplier)
  6. Score      -> absolute weighted score                 (per supplier)
  7. Benchmark  -> best score per criterion across suppliers
  8. Rank       -> PPI, tie-breaks, sequential rank
  9. Persist    -> write to SQLite under one rfp_run_id
  10. Present   -> caller (Streamlit) renders the returned structure
"""

import uuid
import json
from datetime import datetime, timezone

from tools import pdf_tool, llm_tool, validation_tool, ranking_tool
from database.db_setup import get_connection


def create_run() -> str:
    """Step 3: Batch. Creates a new rfp_runs row and returns its id."""
    run_id = str(uuid.uuid4())
    conn = get_connection()
    conn.execute(
        "INSERT INTO rfp_runs (rfp_run_id, created_at, status) VALUES (?, ?, ?)",
        (run_id, datetime.now(timezone.utc).isoformat(), "in_progress"),
    )
    conn.commit()
    conn.close()
    return run_id


def run_batch_evaluation(supplier_inputs: list, active_criteria: list,
                          model: str = None, api_key: str = None, base_url: str = None,
                          max_tokens: int = None, provider: str = None,
                          progress_callback=None) -> dict:
    """
    supplier_inputs: list of dicts:
        {
          "supplier_name": str,
          "submission_date": "YYYY-MM-DD",
          "experience_rating": float (e.g. 0-10),
          "pdf_bytes": bytes  (raw uploaded PDF content)
        }
    active_criteria: list of dicts from database (criterion_id, name, weight, max_score)
    model/api_key/base_url/max_tokens: passed straight to
        tools.llm_tool.resolve_provider()/call_llm_live() (OpenRouter-first,
        then Anthropic, then OpenAI; see tools/llm_tool.py for the full
        resolution order). max_tokens caps the LLM's OUTPUT length per
        supplier call (defaults to 800 if not given -- lower it, e.g. 500,
        if you're on a tight token/credit budget). Leave all four None to
        resolve purely from environment variables.
    progress_callback: optional fn(step:int, total:int, message:str) for UI progress bars

    Makes a real LLM call for every supplier -- there is no offline mode.
    Raises ValueError immediately (before creating a run row) if no API key
    resolves via tools.llm_tool.resolve_provider().

    Returns a dict:
        {
          "rfp_run_id": str,
          "created_at": iso str,
          "results": [ per-supplier enriched record, sorted/ranked ],
          "criteria_used": active_criteria,
        }
    """
    total_steps = len(supplier_inputs) + 3  # evaluate-each + benchmark + rank + persist
    step = 0

    def tick(msg):
        nonlocal step
        step += 1
        if progress_callback:
            progress_callback(step, total_steps, msg)

    # Fail fast, before creating a run row or touching the DB, if no LLM
    # provider can be resolved -- avoids a half-created run on a config error.
    llm_tool.resolve_provider(model=model, api_key=api_key, base_url=base_url, provider=provider)

    # Step 3: Batch
    run_id = create_run()

    # Step 4 + 5 + 6: Evaluate, Validate, Score -- per supplier
    all_suppliers_criteria = {}
    per_supplier_meta = {}
    per_supplier_warnings = {}

    for s in supplier_inputs:
        name = s["supplier_name"]
        tick(f"Extracting & evaluating: {name}")

        proposal_text = pdf_tool.extract_text_from_pdf(s["pdf_bytes"])
        raw_json = llm_tool.evaluate_supplier(
            name, proposal_text, active_criteria,
            model=model, api_key=api_key, base_url=base_url, max_tokens=max_tokens,
            provider=provider,
        )
        # proposal_text is passed through so the Validation Tool can run
        # its evidence-groundedness check (see tools/validation_tool.py) --
        # a cheap, RAG-adjacent-but-not-RAG technique that flags evidence
        # the LLM claims but that doesn't actually appear in the document.
        normalized, warnings = validation_tool.validate_and_normalize(
            raw_json, active_criteria, name, proposal_text=proposal_text,
        )

        all_suppliers_criteria[name] = normalized
        per_supplier_warnings[name] = warnings
        per_supplier_meta[name] = {
            "submission_date": s["submission_date"],
            "experience_rating": float(s["experience_rating"]),
            "raw_llm_response": raw_json,
        }

    # Step 7: Benchmark (needs all suppliers' scores first)
    tick("Calculating peer benchmarks")
    benchmarks = ranking_tool.compute_benchmarks(all_suppliers_criteria)

    # Step 8: Rank (compute per-supplier enriched criteria, PPI, absolute score, then sort)
    tick("Scoring, computing PPI, applying tie-breaks")
    supplier_records = []
    for name, normalized in all_suppliers_criteria.items():
        enriched = ranking_tool.compute_gaps_and_relative(normalized, benchmarks)
        absolute_score = ranking_tool.compute_absolute_score(normalized)
        ppi = ranking_tool.compute_ppi(enriched)

        supplier_records.append({
            "supplier_name": name,
            "submission_date": per_supplier_meta[name]["submission_date"],
            "experience_rating": per_supplier_meta[name]["experience_rating"],
            "absolute_score": absolute_score,
            "ppi": ppi,
            "criteria": enriched,
            "warnings": per_supplier_warnings[name],
            "raw_llm_response": per_supplier_meta[name]["raw_llm_response"],
        })

    ranked = ranking_tool.rank_suppliers(supplier_records)

    # Step 9: Persist
    tick("Persisting results to SQLite")
    conn = get_connection()
    for r in ranked:
        result_json = json.dumps(r)
        conn.execute(
            """INSERT INTO supplier_results
               (rfp_run_id, supplier_name, submission_date, experience_rating,
                absolute_score, ppi, final_rank, result_json)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (run_id, r["supplier_name"], r["submission_date"], r["experience_rating"],
             r["absolute_score"], r["ppi"], r["final_rank"], result_json),
        )
    conn.execute("UPDATE rfp_runs SET status = ? WHERE rfp_run_id = ?", ("completed", run_id))
    conn.commit()
    conn.close()

    return {
        "rfp_run_id": run_id,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "results": ranked,
        "criteria_used": active_criteria,
        "engine": "direct",
    }


def load_run_from_db(run_id: str) -> dict:
    """Step 10 helper: reload a persisted run for display / re-download."""
    conn = get_connection()
    run_row = conn.execute(
        "SELECT * FROM rfp_runs WHERE rfp_run_id = ?", (run_id,)
    ).fetchone()
    if not run_row:
        conn.close()
        return None

    rows = conn.execute(
        "SELECT * FROM supplier_results WHERE rfp_run_id = ? ORDER BY final_rank",
        (run_id,),
    ).fetchall()
    conn.close()

    results = [json.loads(r["result_json"]) for r in rows]
    return {
        "rfp_run_id": run_id,
        "created_at": run_row["created_at"],
        "status": run_row["status"],
        "results": results,
    }


def list_runs() -> list:
    conn = get_connection()
    rows = conn.execute(
        "SELECT rfp_run_id, created_at, status FROM rfp_runs ORDER BY created_at DESC"
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]


### 4b. LangGraph engine

```
start_batch → evaluate ─┐
                 ▲       │  (loops once per remaining supplier)
                 └───────┘
                         │
                         ▼
              benchmark_and_rank → persist → END
```

`evaluate` is the only node that calls an LLM. `benchmark_and_rank` and
`persist` only ever see already-validated, normalized data — never a raw
LLM response. Worth noting: unlike the AutoGen version this replaced
(whose `ToolExecutor` ran fixed code through a sandboxed
`code_execution_config` *specifically to demonstrate* that the
validation/ranking logic wasn't LLM-authored), LangGraph nodes are just
plain Python functions the graph engine calls directly — the
non-authorship guarantee holds by construction here, no separate sandbox
step needed to prove it.

**🐛 Bug found during testing, not by reading the code.** First version of
the progress callback ticked step `9` at the start of
`benchmark_and_rank` and step `total` (11, for a 4-supplier batch) at
`persist` — skipping step 10 entirely. Totally cosmetic (the progress bar
just jumped from ~82% to 100%), but it bugged me, so I added an explicit
tick at the end of `benchmark_and_rank` (step 10) so the sequence runs
1→11 with no gaps. Caught this by actually logging every step number a
real run produced and asserting they were consecutive — reading the code
alone, the off-by-one wasn't obvious.

In [ ]:
%%writefile vendorscope/agents_langgraph/__init__.py


In [ ]:
# --- ENGINE #2, ATTEMPT 1: AutoGen. Built, working, then replaced. ---
# Rough sketch of what it looked like (not the full file -- this never
# shipped in this form, kept here as a comment for reference):
#
# import autogen
#
# evaluator = autogen.AssistantAgent(
#     name="Evaluator", llm_config=False,   # replies produced by a custom hook below
# )
# executor = autogen.UserProxyAgent(
#     name="ToolExecutor",
#     code_execution_config={"work_dir": "coding", "use_docker": False},
# )
#
# def custom_reply(recipient, messages, sender, config):
#     task = json.loads(messages[-1]["content"])
#     # the ONLY line in this whole engine that calls an LLM:
#     raw_json = llm_tool.evaluate_supplier(task["supplier_name"], task["proposal_text"], ...)
#     return True, raw_json
# evaluator.register_reply([autogen.Agent, None], custom_reply)
#
# # validation + ranking run as REAL executed code, specifically to prove
# # they aren't LLM-authored:
# executor.execute_code_blocks([("python", fixed_validation_and_ranking_code)])
#
# This worked correctly -- I'm not describing a bug, I'm describing
# something I later swapped out for a different reason (see the markdown
# above): the brief's own 10-step diagram is already a directed graph with
# one loop, and LangGraph models that shape directly. AutoGen models a
# *conversation between agents*, which is a slightly indirect way to
# represent what's really just sequential data processing with one LLM
# call in it. Both are legitimate "agent framework" implementations per
# the brief's own suggested-tools list -- this was a design preference,
# not a correctness fix.
#
# --- ENGINE #2, WHAT SHIPPED: LangGraph (see the %%writefile cell below) ---
print("(no-op cell -- see agents_langgraph/langgraph_pipeline.py below for what shipped)")


In [ ]:
%%writefile vendorscope/agents_langgraph/langgraph_pipeline.py
"""
langgraph_pipeline.py
-----------------------
Same 10-step pipeline as agents/orchestrator.py, re-implemented as an
explicit LangGraph StateGraph instead of plain Python function calls.

Why LangGraph fits this brief well: the project's own architecture diagram
(Setup -> Input -> Batch -> Evaluate -> Validate -> Score -> Benchmark ->
Rank -> Persist -> Present) is already a directed graph with one loop
(evaluate-then-validate, once per supplier). LangGraph models exactly
that shape -- typed state passed node to node, with an explicit
conditional edge for the per-supplier loop -- so this engine is a fairly
literal implementation of the brief's own diagram, not a generic
"agentic wrapper" bolted on top of it.

Nodes:
  - start_batch          : creates the rfp_runs row, initializes state
  - evaluate              : Document Tool (extract PDF text) + Evaluation
                             Agent (real LLM call) + Validation Tool, for
                             ONE supplier; loops back to itself via a
                             conditional edge until every supplier has been
                             evaluated
  - benchmark_and_rank   : Ranking Tool -- benchmarks, absolute score, PPI,
                             tie-breaks, sequential rank (fully deterministic)
  - persist               : writes the ranked results to SQLite

Unlike the earlier AutoGen engine's ToolExecutor (which ran fixed code
through a sandboxed code_execution_config specifically to demonstrate that
validation/ranking logic isn't LLM-authored), LangGraph nodes are just
plain Python functions we wrote and the graph engine calls directly -- the
same non-authorship guarantee holds by construction, with no separate
sandbox-execution step needed to demonstrate it. The "evaluate" node is
the ONLY node that calls an LLM (tools.llm_tool.evaluate_supplier());
benchmark_and_rank and persist only ever call tools.ranking_tool /
database.db_setup, which never see a raw LLM response, only already-
validated, normalized data.

Requirements: pip install langgraph
"""

import uuid
import json
from datetime import datetime, timezone
from typing import TypedDict, Optional, Callable, Any

from langgraph.graph import StateGraph, END

from tools import pdf_tool, llm_tool, validation_tool, ranking_tool
from database.db_setup import get_connection


class GraphState(TypedDict):
    # ---- set once at invoke time, read-only from every node's perspective ----
    supplier_inputs: list
    active_criteria: list
    model: Optional[str]
    api_key: Optional[str]
    base_url: Optional[str]
    max_tokens: Optional[int]
    provider: Optional[str]
    progress_callback: Optional[Callable[[int, int, str], Any]]

    # ---- working state, updated node to node ----
    run_id: str
    supplier_index: int
    all_suppliers_criteria: dict
    per_supplier_meta: dict
    per_supplier_warnings: dict

    # ---- final output ----
    ranked: list


def _tick(state: GraphState, step: int, total: int, msg: str):
    cb = state.get("progress_callback")
    if cb:
        cb(step, total, msg)


def _total_steps(state: GraphState) -> int:
    return len(state["supplier_inputs"]) * 2 + 3


def node_start_batch(state: GraphState) -> dict:
    """Step 3: Batch. Creates a new rfp_runs row and initializes the
    per-supplier accumulators the evaluate loop will fill in."""
    run_id = str(uuid.uuid4())
    conn = get_connection()
    conn.execute(
        "INSERT INTO rfp_runs (rfp_run_id, created_at, status) VALUES (?, ?, ?)",
        (run_id, datetime.now(timezone.utc).isoformat(), "in_progress"),
    )
    conn.commit()
    conn.close()
    return {
        "run_id": run_id,
        "supplier_index": 0,
        "all_suppliers_criteria": {},
        "per_supplier_meta": {},
        "per_supplier_warnings": {},
    }


def node_evaluate_one_supplier(state: GraphState) -> dict:
    """Steps 4+5: Evaluate (Document Tool + Evaluation Agent -- the ONLY
    real LLM call in this graph) and Validate (Validation Tool, including
    the evidence-groundedness check), for exactly one supplier per visit
    to this node. The conditional edge below re-enters this same node
    until every supplier has been processed."""
    idx = state["supplier_index"]
    s = state["supplier_inputs"][idx]
    name = s["supplier_name"]
    active_criteria = state["active_criteria"]
    total = _total_steps(state)

    _tick(state, idx * 2 + 1, total, f"[LangGraph] Evaluator scoring: {name}")
    proposal_text = pdf_tool.extract_text_from_pdf(s["pdf_bytes"])
    raw_json = llm_tool.evaluate_supplier(
        name, proposal_text, active_criteria,
        model=state.get("model"), api_key=state.get("api_key"),
        base_url=state.get("base_url"), max_tokens=state.get("max_tokens"),
        provider=state.get("provider"),
    )

    _tick(state, idx * 2 + 2, total, f"[LangGraph] Validating: {name}")
    normalized, warnings = validation_tool.validate_and_normalize(
        raw_json, active_criteria, name, proposal_text=proposal_text,
    )

    all_criteria = dict(state["all_suppliers_criteria"])
    all_criteria[name] = normalized
    per_meta = dict(state["per_supplier_meta"])
    per_meta[name] = {
        "submission_date": s["submission_date"],
        "experience_rating": float(s["experience_rating"]),
        "raw_llm_response": raw_json,
    }
    per_warn = dict(state["per_supplier_warnings"])
    per_warn[name] = warnings

    return {
        "supplier_index": idx + 1,
        "all_suppliers_criteria": all_criteria,
        "per_supplier_meta": per_meta,
        "per_supplier_warnings": per_warn,
    }


def route_after_evaluate(state: GraphState) -> str:
    """Conditional edge: loop back to 'evaluate' while suppliers remain,
    otherwise proceed to benchmarking/ranking."""
    if state["supplier_index"] < len(state["supplier_inputs"]):
        return "evaluate"
    return "benchmark_and_rank"


def node_benchmark_and_rank(state: GraphState) -> dict:
    """Steps 6-8: Score, Benchmark, Rank -- entirely deterministic Python
    via tools/ranking_tool.py. Never touches a raw LLM response, only the
    already-validated, normalized criteria produced by the evaluate node."""
    total = _total_steps(state)
    _tick(state, len(state["supplier_inputs"]) * 2 + 1, total,
          "[LangGraph] Calculating peer benchmarks, scoring, ranking")

    all_suppliers_criteria = state["all_suppliers_criteria"]
    per_supplier_meta = state["per_supplier_meta"]
    per_supplier_warnings = state["per_supplier_warnings"]

    benchmarks = ranking_tool.compute_benchmarks(all_suppliers_criteria)

    supplier_records = []
    for name, normalized in all_suppliers_criteria.items():
        enriched = ranking_tool.compute_gaps_and_relative(normalized, benchmarks)
        absolute_score = ranking_tool.compute_absolute_score(normalized)
        ppi = ranking_tool.compute_ppi(enriched)
        supplier_records.append({
            "supplier_name": name,
            "submission_date": per_supplier_meta[name]["submission_date"],
            "experience_rating": per_supplier_meta[name]["experience_rating"],
            "absolute_score": absolute_score,
            "ppi": ppi,
            "criteria": enriched,
            "warnings": per_supplier_warnings[name],
            "raw_llm_response": per_supplier_meta[name]["raw_llm_response"],
        })

    ranked = ranking_tool.rank_suppliers(supplier_records)
    _tick(state, len(state["supplier_inputs"]) * 2 + 2, total,
          "[LangGraph] Benchmarking, scoring, and ranking complete")
    return {"ranked": ranked}


def node_persist(state: GraphState) -> dict:
    """Step 9: Persist. Writes the ranked results to SQLite under the
    run's rfp_run_id."""
    total = _total_steps(state)
    _tick(state, total, total, "[LangGraph] Persisting results to SQLite")

    run_id = state["run_id"]
    ranked = state["ranked"]
    conn = get_connection()
    for r in ranked:
        conn.execute(
            """INSERT INTO supplier_results
               (rfp_run_id, supplier_name, submission_date, experience_rating,
                absolute_score, ppi, final_rank, result_json)
               VALUES (?, ?, ?, ?, ?, ?, ?, ?)""",
            (run_id, r["supplier_name"], r["submission_date"], r["experience_rating"],
             r["absolute_score"], r["ppi"], r["final_rank"], json.dumps(r)),
        )
    conn.execute("UPDATE rfp_runs SET status = ? WHERE rfp_run_id = ?", ("completed", run_id))
    conn.commit()
    conn.close()
    return {}


def build_graph():
    """Builds and compiles the StateGraph. Exposed separately from
    run_langgraph_batch_evaluation() so the graph structure can be
    inspected/visualized independently of running it."""
    g = StateGraph(GraphState)
    g.add_node("start_batch", node_start_batch)
    g.add_node("evaluate", node_evaluate_one_supplier)
    g.add_node("benchmark_and_rank", node_benchmark_and_rank)
    g.add_node("persist", node_persist)

    g.set_entry_point("start_batch")
    g.add_edge("start_batch", "evaluate")
    g.add_conditional_edges(
        "evaluate", route_after_evaluate,
        {"evaluate": "evaluate", "benchmark_and_rank": "benchmark_and_rank"},
    )
    g.add_edge("benchmark_and_rank", "persist")
    g.add_edge("persist", END)

    return g.compile()


def run_langgraph_batch_evaluation(supplier_inputs: list, active_criteria: list,
                                    model: str = None, api_key: str = None,
                                    base_url: str = None, max_tokens: int = None,
                                    provider: str = None,
                                    progress_callback=None) -> dict:
    """
    Runs the full 10-step pipeline through the LangGraph engine, always
    with a real LLM call for every supplier (see tools/llm_tool.py -- there
    is no offline mode anywhere in this project). If no API key resolves,
    raises a ValueError before creating a run row or touching the database.

    Same signature/return shape as agents.orchestrator.run_batch_evaluation,
    so app.py can call either engine with matching inputs/outputs.
    """
    # Fail fast, before creating a run row or touching the DB, if no live
    # provider can be resolved -- avoids a half-created run on a config error.
    llm_tool.resolve_provider(model=model, api_key=api_key, base_url=base_url, provider=provider)

    graph = build_graph()
    initial_state: GraphState = {
        "supplier_inputs": supplier_inputs,
        "active_criteria": active_criteria,
        "model": model,
        "api_key": api_key,
        "base_url": base_url,
        "max_tokens": max_tokens,
        "provider": provider,
        "progress_callback": progress_callback,
        "run_id": "",
        "supplier_index": 0,
        "all_suppliers_criteria": {},
        "per_supplier_meta": {},
        "per_supplier_warnings": {},
        "ranked": [],
    }
    # Default recursion_limit (25) comfortably covers small batches (each
    # supplier costs one step through the evaluate loop), but bump it for
    # safety on larger batches -- this only bounds graph steps, not cost.
    final_state = graph.invoke(initial_state, config={"recursion_limit": 200})

    return {
        "rfp_run_id": final_state["run_id"],
        "created_at": datetime.now(timezone.utc).isoformat(),
        "results": final_state["ranked"],
        "criteria_used": active_criteria,
        "engine": "langgraph",
    }


In [ ]:
# --- Progress-tick bug: found by testing, not by reading the code ---
#
# ATTEMPT 1 (shipped briefly, then fixed -- this is what it looked like):
#
#   def node_benchmark_and_rank(state):
#       total = _total_steps(state)
#       _tick(state, len(state["supplier_inputs"]) * 2 + 1, total, "...")
#       # ... compute benchmarks, scores, ppi, ranked ...
#       return {"ranked": ranked}          # <-- no second tick here
#
#   def node_persist(state):
#       total = _total_steps(state)
#       _tick(state, total, total, "...")  # jumps straight to the final step
#
#   For a 4-supplier batch (total=11): ticks landed at ...,9, then 11.
#   Step 10 was never ticked -- the progress bar visibly jumped from ~82%
#   straight to 100%, skipping a step. Not incorrect, just sloppy.
#
# HOW I ACTUALLY FOUND IT (not from reading the two functions above and
# spotting the gap -- I didn't, at first):
#
#   steps_seen = []
#   run_langgraph_batch_evaluation(..., progress_callback=lambda s,t,m: steps_seen.append(s))
#   assert steps_seen == list(range(1, total + 1))   # <-- this failed
#
# FIX (what shipped -- see node_benchmark_and_rank in the file above):
#   added a second _tick(...) call at the END of node_benchmark_and_rank,
#   right after `ranked = ranking_tool.rank_suppliers(...)`, using step
#   number `len(supplier_inputs) * 2 + 2`. That closes the gap: steps now
#   run 1, 2, 3, ..., 11 with nothing skipped.
print("(no-op cell -- see the langgraph_pipeline.py source above for the actual fix)")


## 5. The validation/error-case demo

This grew incrementally as the error types above got added:

1. Started with just the out-of-range score (matches the brief's clipping example).
2. Added the missing-criterion case once that path existed in `validation_tool.py`.
3. Added the fabricated-evidence case last, once `is_evidence_grounded()` existed —
   this one needed a *real* proposal excerpt to check against, not just a
   made-up JSON blob, so the demo constructs a short real-sounding excerpt
   and a scorecard whose evidence for one criterion doesn't match it.

All three are asserted at the end, so this cell either proves the safety
net works or fails loudly — no silent "looks about right."

In [ ]:
%%writefile vendorscope/sample_data/generate_supplier_pdfs.py
"""
generate_supplier_pdfs.py
Generates four fictional supplier RFP response PDFs used for testing and
demoing the app. Run: python sample_data/generate_supplier_pdfs.py
No real/confidential supplier data is used -- entirely synthetic.
"""

import os
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
from reportlab.lib.units import inch

OUT_DIR = os.path.join(os.path.dirname(__file__), "supplier_pdfs")
os.makedirs(OUT_DIR, exist_ok=True)

styles = getSampleStyleSheet()


def section(title, body_paragraphs, story):
    story.append(Paragraph(title, styles["Heading2"]))
    for p in body_paragraphs:
        story.append(Paragraph(p, styles["Normal"]))
        story.append(Spacer(1, 6))
    story.append(Spacer(1, 10))


def price_table(rows, story):
    data = [["Item", "Detail", "Cost (USD)"]] + rows
    t = Table(data, colWidths=[1.8 * inch, 3 * inch, 1.5 * inch])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2c3e50")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
    ]))
    story.append(t)
    story.append(Spacer(1, 12))


def build_pdf(filename, title, sections, prices):
    path = os.path.join(OUT_DIR, filename)
    doc = SimpleDocTemplate(path, pagesize=letter,
                             topMargin=0.7 * inch, bottomMargin=0.7 * inch)
    story = [Paragraph(title, styles["Title"]), Spacer(1, 14)]
    for sec_title, paras in sections:
        section(sec_title, paras, story)
    story.append(Paragraph("Price Table", styles["Heading2"]))
    price_table(prices, story)
    doc.build(story)
    print(f"Created {path}")


# ---------------------------------------------------------------------------
# Apex Systems: strong technical + security, higher price, moderate schedule
# ---------------------------------------------------------------------------
build_pdf(
    "Apex_Systems.pdf",
    "RFP Response: Apex Systems",
    [
        ("Executive Summary", [
            "Apex Systems proposes a modular, cloud-native platform architecture designed "
            "for high scalability and long-term maintainability, addressing the client's "
            "requirement for a resilient procurement automation system.",
        ]),
        ("Proposed Solution & Architecture", [
            "Our architecture uses a microservices design with independent scaling of "
            "the ingestion, scoring, and reporting layers. Integrations are exposed via "
            "documented REST APIs with OAuth2, supporting both synchronous and event-driven "
            "consumption patterns.",
            "The system is built for horizontal scalability using container orchestration, "
            "with load-tested throughput exceeding 500 requests/second at P95 latency under 200ms.",
        ]),
        ("Timeline, Team Structure & Milestones", [
            "Delivery is organized into four phases across 16 weeks: Discovery (2 weeks), "
            "Core Build (8 weeks), Integration & Hardening (4 weeks), and Go-Live (2 weeks). "
            "A dedicated team of 6 (1 architect, 3 engineers, 1 QA, 1 PM) is staffed for the "
            "full engagement, with weekly milestone checkpoints and a documented risk log.",
        ]),
        ("Security, Compliance & Risk Controls", [
            "Apex maintains SOC 2 Type II and ISO 27001 certifications. All data is encrypted "
            "at rest (AES-256) and in transit (TLS 1.3). Role-based access control, audit "
            "logging, and quarterly third-party penetration testing are standard. A named "
            "compliance officer is assigned to every engagement for auditability.",
        ]),
        ("Support Model, Experience & References", [
            "24/7 tiered support (L1-L3) with a 1-hour SLA for critical incidents. Apex has "
            "delivered 3 comparable procurement-automation platforms for enterprise clients "
            "in the last 4 years, with references available on request.",
        ]),
        ("Risks", [
            "Primary risk is the 16-week schedule being tight if client-side integration "
            "credentials are delayed beyond week 2; mitigation includes a buffer sprint.",
        ]),
    ],
    prices=[
        ["Platform License", "Annual, includes updates", "58,000"],
        ["Implementation", "One-time, phases 1-4", "42,000"],
        ["Support (Year 1)", "24/7 tiered, included Y1", "0 (bundled)"],
        ["Total (Year 1)", "License + implementation", "100,000"],
    ],
)

# ---------------------------------------------------------------------------
# BrightPath Tech: lowest price, fastest timeline, weak compliance/experience
# ---------------------------------------------------------------------------
build_pdf(
    "BrightPath_Tech.pdf",
    "RFP Response: BrightPath Tech",
    [
        ("Executive Summary", [
            "BrightPath Tech offers a lean, cost-effective solution that can be delivered "
            "quickly, ideal for teams wanting to move fast without a large upfront budget.",
        ]),
        ("Proposed Solution & Approach", [
            "We will build a single-service web application connecting to the client's "
            "existing SQL database. The system will expose a basic REST endpoint for "
            "data upload and a dashboard for results.",
        ]),
        ("Timeline, Team & Milestones", [
            "We can deliver the full solution in 6 weeks using a small team of 2 developers. "
            "Milestones: Week 2 - basic upload flow; Week 4 - scoring logic; Week 6 - dashboard "
            "and handover.",
        ]),
        ("Security & Compliance", [
            "Standard HTTPS is used for all traffic. Passwords are hashed. We are happy to "
            "discuss additional compliance requirements if the client can specify them.",
        ]),
        ("Support & Experience", [
            "Email support is available during business hours (9am-6pm). This would be our "
            "first project specifically in the procurement-evaluation space, though we have "
            "built several internal dashboards for other clients.",
        ]),
        ("Pricing Notes", [
            "Pricing assumes the client provides sample data in a ready-to-use CSV/JSON "
            "format; additional data-cleaning work would be billed separately at $80/hour.",
        ]),
    ],
    prices=[
        ["Development", "One-time, 6-week build", "18,000"],
        ["Hosting", "Annual", "2,400"],
        ["Support (Year 1)", "Business hours, email only", "3,000"],
        ["Total (Year 1)", "Dev + hosting + support", "23,400"],
    ],
)

# ---------------------------------------------------------------------------
# NexaWorks: balanced, strongest implementation plan and support model
# ---------------------------------------------------------------------------
build_pdf(
    "NexaWorks.pdf",
    "RFP Response: NexaWorks",
    [
        ("Executive Summary", [
            "NexaWorks proposes a balanced solution that combines solid technical design "
            "with a highly structured implementation approach and a support model built "
            "around long-term partnership.",
        ]),
        ("Proposed Solution & Architecture", [
            "The platform uses a service-oriented architecture with a document-processing "
            "pipeline, a rules engine for deterministic scoring, and a reporting layer. "
            "Integrations are available via REST API and scheduled batch export.",
        ]),
        ("Implementation Plan", [
            "Our implementation methodology follows a detailed 5-phase plan: Discovery (1 wk), "
            "Design Sign-off (1 wk), Build Sprint 1-3 (6 wks), UAT (2 wks), Go-Live & Hypercare "
            "(2 wks) — 12 weeks total. Each phase has named owners, entry/exit criteria, and a "
            "RAID log reviewed weekly with the client's steering committee. Staffing includes "
            "1 delivery lead, 2 engineers, 1 QA, and a dedicated client success manager from day one.",
        ]),
        ("Security & Compliance", [
            "Data is encrypted in transit (TLS 1.2+) and at rest. We follow a documented "
            "internal security checklist aligned to common industry frameworks and can "
            "pursue formal certification alongside the client if required.",
        ]),
        ("Support Model, Experience & References", [
            "Dedicated named support contact plus a ticketing portal, 4-hour response SLA "
            "during business hours and next-business-day for non-critical items. NexaWorks "
            "has delivered 5 similar vendor-scoring and evaluation platforms across logistics "
            "and healthcare clients in the past 3 years; two references are available.",
        ]),
        ("Risks", [
            "Integration risk exists if client legacy systems lack a documented API; a "
            "discovery-phase spike is included specifically to de-risk this.",
        ]),
    ],
    prices=[
        ["Platform Build", "One-time, 12-week plan", "36,000"],
        ["Annual Support", "4-hour SLA, dedicated CSM", "9,600"],
        ["Hosting", "Annual, managed cloud", "3,600"],
        ["Total (Year 1)", "Build + support + hosting", "49,200"],
    ],
)

# ---------------------------------------------------------------------------
# Orbit Digital: strong experience/references, vague integration, medium price
# ---------------------------------------------------------------------------
build_pdf(
    "Orbit_Digital.pdf",
    "RFP Response: Orbit Digital",
    [
        ("Executive Summary", [
            "Orbit Digital brings deep domain experience in procurement technology, having "
            "delivered similar systems for a range of enterprise and mid-market clients over "
            "the past decade.",
        ]),
        ("Proposed Solution", [
            "We will build a web-based evaluation tool that connects to the client's data "
            "sources and produces supplier rankings. Specific integration mechanisms will "
            "be finalized during the discovery phase based on the client's existing systems.",
        ]),
        ("Timeline & Team", [
            "Estimated delivery is 10-14 weeks depending on discovery findings. Our senior "
            "delivery team has an average of 8 years of experience in enterprise procurement "
            "software.",
        ]),
        ("Security & Compliance", [
            "Orbit Digital is ISO 27001 certified at the company level. Standard encryption "
            "and access-control practices are applied to all client engagements; certificate "
            "and audit documentation is available on request.",
        ]),
        ("Support Model, Experience & References", [
            "Orbit Digital has completed 7 comparable supplier-evaluation projects over the "
            "past 9 years, including for two Fortune 500 procurement teams. Three client "
            "references with contact details are available on request. Support is offered "
            "via a dedicated account manager with a 24-hour response SLA.",
        ]),
        ("Risks", [
            "Because the exact integration approach depends on discovery-phase findings, "
            "final architecture details and timeline may shift after week 2.",
        ]),
    ],
    prices=[
        ["Discovery & Design", "2-3 weeks", "12,000"],
        ["Build & Integration", "8-11 weeks, scope TBD post-discovery", "38,000"],
        ["Annual Support", "24-hour SLA, dedicated AM", "7,200"],
        ["Total (Year 1, est.)", "Discovery + build + support", "57,200"],
    ],
)

print("All 4 synthetic supplier PDFs generated.")


In [ ]:
%%writefile vendorscope/demo_validation_error_case.py
"""
demo_validation_error_case.py
------------------------------
Standalone demonstration of tools/validation_tool.py's error-handling
path, satisfying the project brief's requirement for "at least one
validation/error case" demonstration -- independent of any LLM call, so
it needs no API key and always produces the same output.

This does NOT simulate an LLM. It constructs a deliberately malformed JSON
string -- three kinds of mistake a real LLM can plausibly make -- and
shows validate_and_normalize() catching all three before they ever reach
the Ranking Tool:

  1. An out-of-range score (15 out of a max of 10).
  2. A missing criterion (the LLM simply didn't return one).
  3. Fabricated evidence -- a claimed quote that does not actually appear
     anywhere in the supplier's proposal text (the evidence-groundedness
     check; see is_evidence_grounded() in tools/validation_tool.py).

This is a direct stress-test of the safety net, which is arguably a
clearer demonstration than waiting for a live model to happen to make a
mistake.

Run: python demo_validation_error_case.py
"""

import json
from database.db_setup import init_db, get_active_criteria
from tools import validation_tool

REAL_PROPOSAL_EXCERPT = """
Apex Systems proposes a modular, cloud-native platform architecture
designed for high scalability and long-term maintainability. Our
architecture uses a microservices design with independent scaling of the
ingestion, scoring, and reporting layers. Delivery is organized into four
phases across 16 weeks: Discovery, Core Build, Integration & Hardening,
and Go-Live. Apex maintains SOC 2 Type II and ISO 27001 certifications,
with role-based access control and quarterly penetration testing.
"""


def main():
    init_db()
    active_criteria = get_active_criteria()

    print("=== Active criteria (source of truth for validation) ===")
    for c in active_criteria:
        print(f"  id={c['criterion_id']}  {c['name']!r}  weight={c['weight']}%  max_score={c['max_score']}")
    print()

    # A deliberately malformed "raw LLM response" -- three kinds of mistake
    # a real model can plausibly make:
    #   1. "Security & Compliance" (criterion_id=4) scored 15, above its
    #      max_score of 10.
    #   2. "Support & Experience" (criterion_id=5) is missing entirely.
    #   3. "Commercial Value" (criterion_id=3)'s evidence is fabricated --
    #      it claims a specific pricing detail that never appears in the
    #      real proposal excerpt above.
    malformed_raw_response = json.dumps({
        "supplier_name": "Demo Supplier",
        "criteria": [
            {"criterion_id": 1, "score": 8, "max_score": 10,
             "justification": "Strong architecture description.",
             "evidence": "Microservices design with independent scaling of the ingestion, scoring, and reporting layers."},
            {"criterion_id": 2, "score": 7, "max_score": 10,
             "justification": "Clear phased plan.",
             "evidence": "Four-phase delivery schedule across 16 weeks."},
            {"criterion_id": 3, "score": 6, "max_score": 10,
             "justification": "Pricing itemized but high.",
             "evidence": "Total year-1 cost of $250,000 with a 10% early-payment discount."},  # FABRICATED
            {"criterion_id": 4, "score": 15, "max_score": 10,  # OUT OF RANGE
             "justification": "Certifications look strong.",
             "evidence": "SOC 2 Type II and ISO 27001 certifications mentioned."},
            # criterion_id 5 is MISSING entirely
        ],
        "risks": ["Schedule risk if credentials are delayed"],
        "overall_summary": "A strong proposal overall.",
    })

    print("=== Deliberately malformed input (simulating three real LLM mistakes) ===")
    print(malformed_raw_response)
    print()

    normalized, warnings = validation_tool.validate_and_normalize(
        malformed_raw_response, active_criteria, "Demo Supplier",
        proposal_text=REAL_PROPOSAL_EXCERPT,
    )

    print("=== Validation Tool warnings raised ===")
    if not warnings:
        print("  (none -- this should not happen for this deliberately broken input)")
    for w in warnings:
        print(f"  - {w}")
    print()

    print("=== Normalized output the Ranking Tool actually receives ===")
    for c in normalized:
        print(f"  {c['name']:25s} score={c['score']:>5}/{c['max_score']}  "
              f"(clipped/defaulted/flagged safely, never exceeds max_score)")
    print()

    assert len(warnings) == 3, f"Expected exactly 3 warnings, got {len(warnings)}: {warnings}"
    assert any("exceeded max_score" in w for w in warnings), "Expected an out-of-range warning"
    assert any("missing from LLM output" in w for w in warnings), "Expected a missing-criterion warning"
    assert any("not appear to be grounded" in w for w in warnings), "Expected an evidence-groundedness warning"
    assert all(c["score"] <= c["max_score"] for c in normalized), "A score still exceeds max_score!"

    print("CONFIRMED: all three deliberate errors were caught and safely handled --")
    print("the out-of-range score was clipped to max_score, the missing criterion")
    print("was defaulted to 0 and flagged, and the fabricated evidence was flagged")
    print("as not grounded in the supplier's actual proposal text. None of these")
    print("bad values reach tools/ranking_tool.py unmodified/unflagged.")


if __name__ == "__main__":
    main()


### 🐛 Bugs from building the Streamlit front-end (not shown in this notebook — it only covers pipeline logic, not `app.py`)

Worth documenting anyway, since these are exactly the kind of thing you'd
hit reusing this pattern:

- **Cube-logo icon generator, attempt 1: geometry bug.** The isometric
  basis vectors (`RIGHT`/`LEFT`/`DOWN`) were unit vectors, but I never
  scaled them by the tile size before using them as per-tile step
  vectors. Result: a 2-pixel speck in the corner of a 350×310 canvas.
  Fixed by scaling the basis vectors by `TILE` up front.
- **Leaderboard "podium" cards, attempt 1: rendered as raw text.** Built
  each card as a multi-line f-string with 8-space-indented HTML inside
  it. Card 1 rendered fine; cards 2 and 3 showed up as literal escaped
  HTML source in a gray monospace box. Root cause: a blank/whitespace-
  only line between cards terminated Streamlit's Markdown "HTML block"
  parsing early, and CommonMark treats a 4+-space-indented line right
  after a blank line as an *indented code block* — which is exactly what
  swallowed cards 2 and 3. Fixed by building each card as a single-line
  string with zero internal newlines.
- **API keys, attempt 1: stored in `os.environ`.** Worked fine locally.
  Would have been a real problem on a public Streamlit Cloud deployment —
  `os.environ` is shared across every concurrent visitor's session on the
  same running app instance, so one visitor's key could leak into or get
  overwritten by another's. Fixed by keeping user-typed keys in
  `st.session_state` (per-browser-session) and passing them explicitly as
  a function argument, never through the process environment.
- **Evaluate-button gating, case-sensitivity bug.** The warning message
  said `"No API key is set..."` (capital N); the code checking whether to
  disable the button looked for `"no API key is set"` (lowercase). Python
  string matching is case-sensitive, so the check silently never matched
  — meaning the button stayed clickable even with no key configured.
  Found by writing a test for the exact gating logic, not by inspection.

## 6. Wiring it up and running it

In [ ]:
import sys, os

# Make the freshly-written vendorscope/ package importable, and cd into it
# so relative paths (sample_data/, database/rfp_evaluation.db, etc.) match
# what every module inside the package expects.
sys.path.insert(0, os.path.abspath("vendorscope"))
os.chdir("vendorscope")

from database.db_setup import init_db, get_active_criteria

init_db()  # safe to call repeatedly -- only seeds criteria if the table is empty
criteria = get_active_criteria()
for c in criteria:
    print(f"{c['name']:25s} weight={c['weight']:>3}%  max_score={c['max_score']}")


In [ ]:
!python sample_data/generate_supplier_pdfs.py

### Build the supplier batch (metadata + PDF bytes)

In [ ]:
# Metadata for the 4 synthetic suppliers -- these dates/ratings are just
# realistic-looking test values, not derived from anything in the PDFs
# (submission_date and experience_rating are business context a real user
# would type in, not something the LLM or PDF text tells you).
pdf_dir = "sample_data/supplier_pdfs"
dates = {"Apex_Systems.pdf": "2026-02-10", "BrightPath_Tech.pdf": "2026-02-05",
         "NexaWorks.pdf": "2026-02-08", "Orbit_Digital.pdf": "2026-02-12"}
exp = {"Apex_Systems.pdf": 8.0, "BrightPath_Tech.pdf": 3.0,
       "NexaWorks.pdf": 7.5, "Orbit_Digital.pdf": 9.0}

supplier_inputs = []
for fname in sorted(os.listdir(pdf_dir)):
    # Read the PDF as raw bytes now, not a file path -- both engines pass
    # bytes around in memory (pdf_tool.extract_text_from_pdf takes bytes),
    # matching how Streamlit's uploader hands you file content too.
    with open(os.path.join(pdf_dir, fname), "rb") as f:
        pdf_bytes = f.read()

    name = fname.rsplit(".", 1)[0].replace("_", " ")  # "Apex_Systems.pdf" -> "Apex Systems"
    supplier_inputs.append({
        "supplier_name": name,
        "submission_date": dates[fname],
        "experience_rating": exp[fname],
        "pdf_bytes": pdf_bytes,
    })

print(f"Loaded {len(supplier_inputs)} supplier proposals")


### Run the validation/error-case demo — no API key needed, always the same output

In [ ]:
!python demo_validation_error_case.py

### Provider setup — required from here on, every call is real

`tools/llm_tool.resolve_provider()` checks `OPENROUTER_API_KEY`, then
`ANTHROPIC_API_KEY`, then `OPENAI_API_KEY`, in that order. One key is
enough.

In [ ]:
import os
from getpass import getpass

def get_secret(name):
    """Colab's userdata secrets first (the key icon in the left sidebar,
    Runtime > secrets) -- this survives across sessions so you don't
    retype your key every time you reopen this notebook. Falls through
    to an interactive prompt only if nothing's stored there."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass  # not running in Colab, or no secret set -- fine, fall through
    return None

# Checked in this exact order -- first one found wins. This mirrors
# tools/llm_tool.resolve_provider()'s own priority order, so whichever key
# you set here is the same one the pipeline will actually pick up.
for env_name, label in [
    ("OPENROUTER_API_KEY", "OpenRouter"),
    ("ANTHROPIC_API_KEY", "Anthropic"),
    ("OPENAI_API_KEY", "OpenAI"),
]:
    val = get_secret(env_name) or os.environ.get(env_name)
    if val:
        os.environ[env_name] = val
        print(f"Using {label} (found {env_name}).")
        break
else:
    # `else` on a `for` loop runs only if the loop completed without `break`
    # -- i.e. none of the three keys were found anywhere. Prompt as a
    # last resort rather than silently failing three steps from now.
    key = getpass("No key found in Colab secrets or env vars.\n"
                   "Paste your OpenRouter API key (openrouter.ai/keys): ")
    os.environ["OPENROUTER_API_KEY"] = key
    print("Using OpenRouter (key entered interactively).")


In [ ]:
from tools import llm_tool

# Resolves config only -- no API call yet. This is exactly the check both
# engines run internally before creating a run row, so a missing key
# fails fast rather than half-creating a run.
resolved = llm_tool.resolve_provider()
print("Resolved provider config (no call made yet):", resolved)
print("Default max_tokens per call:", llm_tool.DEFAULT_MAX_TOKENS)


### One real call, so you can see the structured output actually working

With structured-output enforcement active, this should come back
schema-valid on the first try -- not "usually valid," actually valid.

In [ ]:
sample = supplier_inputs[0]  # Apex Systems -- just picking one to inspect closely
from tools import pdf_tool

# Same extraction call both engines use internally -- nothing special
# happens here that doesn't also happen inside run_batch_evaluation().
proposal_text = pdf_tool.extract_text_from_pdf(sample["pdf_bytes"])

# max_tokens=800 is the module default anyway (DEFAULT_MAX_TOKENS in
# tools/llm_tool.py) -- passed explicitly here just to make the budget
# visible, since this is the number you'd lower to demo a truncated/
# malformed response later (see the Validation Tool notes above).
raw_json = llm_tool.evaluate_supplier(
    sample["supplier_name"], proposal_text, get_active_criteria(), max_tokens=800,
)
print(f"--- Raw LLM response for {sample['supplier_name']!r} ---\n")
print(raw_json)


### Full batch, both engines

Each engine makes its own independent LLM call per supplier -- their
rankings are deliberately NOT compared against each other here, since two
separate calls to the same model aren't guaranteed to return identical
scores. Each engine's own leaderboard is the proof that its pipeline
completed a real run.

In [ ]:
from agents.orchestrator import run_batch_evaluation

def progress(step, total, msg):
    # run_batch_evaluation calls this once per pipeline step (extract+
    # evaluate+validate per supplier, then benchmark, then persist) --
    # just printing here, but this is the exact same callback shape
    # app.py wires up to a real Streamlit progress bar.
    print(f"[{step}/{total}] {msg}")

# No llm_mode, no mock flag -- this always makes a real LLM call per
# supplier. If no API key resolved above, this line raises a clear
# ValueError right here rather than partway through the batch.
direct_result = run_batch_evaluation(supplier_inputs, get_active_criteria(),
                                      max_tokens=800, progress_callback=progress)

print("\nDirect engine RFP_RUN_ID:", direct_result["rfp_run_id"])
for r in direct_result["results"]:
    # r["final_rank"] is only meaningful because rank_suppliers() already
    # applied the full 4-level tie-break sort -- this loop just prints
    # results in the order they come back, already sorted.
    print(f"Rank {r['final_rank']}: {r['supplier_name']:20s} PPI={r['ppi']:6.2f}  Abs={r['absolute_score']:6.2f}  warnings={len(r['warnings'])}")


In [ ]:
from agents_langgraph.langgraph_pipeline import run_langgraph_batch_evaluation

# Same supplier_inputs, same criteria, same progress callback -- the ONLY
# thing that differs from the Direct engine call above is which module
# runs the pipeline. Compare the two RFP_RUN_IDs below: different UUIDs,
# different timestamps -- proof these are two independent runs, not one
# cached result reused twice (see the "why do both engines match" note
# from earlier troubleshooting -- temperature=0 makes matching scores
# expected, not suspicious).
langgraph_result = run_langgraph_batch_evaluation(supplier_inputs, get_active_criteria(),
                                                   max_tokens=800, progress_callback=progress)

print("\nLangGraph engine RFP_RUN_ID:", langgraph_result["rfp_run_id"])
for r in langgraph_result["results"]:
    print(f"Rank {r['final_rank']}: {r['supplier_name']:20s} PPI={r['ppi']:6.2f}  Abs={r['absolute_score']:6.2f}  warnings={len(r['warnings'])}")


### Read the model's actual reasoning, both engines

In [ ]:
def show_top_supplier_evidence(result, engine_label):
    # result["results"] is already sorted by final_rank (rank_suppliers()
    # did that), so index [0] is always the rank-1 supplier -- no need to
    # search or re-sort here.
    pick = result["results"][0]
    print(f"=== [{engine_label}] {pick['supplier_name']} -- Rank {pick['final_rank']}  (PPI={pick['ppi']:.2f}) ===\n")
    for c in pick["criteria"]:
        # justification/evidence are the ONLY free-text fields the LLM
        # produced -- score/weight/benchmark/gap/relative_pct are all
        # numbers computed by tools/ranking_tool.py, not the model.
        print(f"[{c['name']}]  score={c['score']}/{c['max_score']}  weight={c['weight']}%")
        print(f"  Justification: {c['justification']}")
        print(f"  Evidence:      {c['evidence']}")
        print()

show_top_supplier_evidence(direct_result, "Direct")
print()
show_top_supplier_evidence(langgraph_result, "LangGraph")


### Export both completed runs as JSON

In [ ]:
import json

# Two separate files, not one -- keeping Direct's and LangGraph's results
# apart makes it obvious which engine produced which file later, and
# avoids implying they should be compared head-to-head (they shouldn't;
# see the earlier note on why their rankings can legitimately differ).
exports = {
    "sample_output/colab_run_result_direct.json": direct_result,
    "sample_output/colab_run_result_langgraph.json": langgraph_result,
}
for export_path, result in exports.items():
    with open(export_path, "w") as f:
        # default=str handles the one non-JSON-native type in here:
        # datetime objects in created_at get stringified instead of
        # raising a TypeError.
        json.dump(result, f, indent=2, default=str)
    print(f"Exported {result['engine']} engine run to {export_path} ({os.path.getsize(export_path)} bytes)")

try:
    from google.colab import files
    for export_path in exports:
        files.download(export_path)
except Exception:
    # Not running in Colab (e.g. testing locally) -- the files are still
    # on disk at the paths above, just not auto-downloaded to your machine.
    print("(Not running in Colab -- files are saved locally at the paths above.)")


## 7. Zip it up for GitHub

Same caveat as always: this notebook only writes the pipeline-logic files.
`app.py`, `requirements.txt`, `.streamlit/`, and `assets/` (the cube logo)
come from the full project package, not from here -- pull those in
separately before deploying to Streamlit Community Cloud.

In [ ]:
import shutil

# Zip the whole vendorscope/ folder as it stands right now -- includes
# every file the %%writefile cells wrote PLUS the SQLite db and any
# sample_output/*.json this session generated. Does NOT include app.py,
# requirements.txt, .streamlit/, or assets/ -- this notebook only ever
# covers the pipeline logic, not the Streamlit UI or its config.
os.chdir("..")
shutil.make_archive("vendorscope-from-colab", "zip", "vendorscope")

from google.colab import files
files.download("vendorscope-from-colab.zip")
